In [10]:
import cv2
from PIL import Image
import base64
import io


In [ ]:
def _preprocessing_video(video_path: str, desired_fps=1, shortest_edge=224):
    try:
        # Mở video bằng OpenCV
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise Exception("Cannot open video")

        # Lấy thông tin video
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0

        # Tính toán kích thước mới: cạnh nhỏ nhất = 224px, giữ aspect ratio
        min_dimension = min(width, height) if (width and height) else shortest_edge
        scale_factor = shortest_edge / min_dimension
        new_width = int(width * scale_factor)
        new_height = int(height * scale_factor)

        # Đảm bảo kích thước là số chẵn (một số codec/resize yêu cầu)
        new_width = new_width if new_width % 2 == 0 else new_width - 1
        new_height = new_height if new_height % 2 == 0 else new_height - 1

        print(f"Scaling video from {width}×{height} to {new_width}×{new_height}")
        print(f"Video info - FPS: {fps}, Total Frames: {total_frames}")

        # Tính frame skip interval (desired_fps frames per second processed)
        frame_skip = max(1, int(round(fps / desired_fps))) if fps and desired_fps > 0 else 1

        print(f"Processing every {frame_skip} frames (approx {desired_fps} fps)...")

        frame_count = 0
        processed_frames = 0
        processed_b64 = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Xử lý mỗi frame_skip frame
            if frame_count % frame_skip == 0:
                processed_frames += 1

                # Resize frame về kích thước mới
                resized_frame = cv2.resize(frame, (new_width, new_height), interpolation=cv2.INTER_LINEAR)

                # Convert từ BGR (OpenCV) sang RGB (PIL/CLIP)
                rgb_frame = cv2.cvtColor(resized_frame, cv2.COLOR_BGR2RGB)

                # Convert numpy array thành PIL Image
                pil_image = Image.fromarray(rgb_frame)

                # Chuyển PIL Image sang base64 string (JPEG)
                buf = io.BytesIO()
                pil_image.save(buf, format='JPEG', quality=100, optimize=True)
                b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
                processed_b64.append(b64)

            frame_count += 1

        cap.release()
        print(f"Processed {processed_frames} frames out of {total_frames} total frames")

        return processed_b64

    except Exception as e:
        print(f"Error processing video with CLIP: {e}")
        return None


In [14]:
video_path = "C:\\Users\\likgn\\Downloads\\2932301-uhd_4096_2160_24fps.mp4"
images = _process_video_with_clip(video_path=video_path, desired_fps=5)

Scaling video from 2732×1440 to 424×224
Video info - FPS: 24.0, Total Frames: 367
Processing every 5 frames (approx 5 fps)...
Processed 74 frames out of 367 total frames
